In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


In [2]:
# implementación de capa lineal como nueronas sparse
class LearnedSparseLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True, hard_threshold:float=0.5):
        super().__init__()
        self.in_features = int(in_features)
        self.out_features = int(out_features)

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)

        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

        # Logits de conectividad (aprendibles): esto ES tu “grafo K” en forma continua
        # Inicializamos con media 0.0 para estar en el punto crítico de binarización (sigmoid(0)=0.5)
        self.logits = nn.Parameter(torch.zeros(out_features, in_features).normal_(mean=0.0, std=0.1))

        # Hiperparámetros internos
        self.temperature = 1.0     # menor -> gates más cercanos a 0/1
        self.hard_threshold = hard_threshold  # umbral para binarizar

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != self.in_features:
            raise ValueError(f"x debe ser (B, {self.in_features})")

        gate_soft = torch.sigmoid(self.logits / self.temperature)          # (m, n)
        gate_hard = (gate_soft > self.hard_threshold).to(x.dtype)          # (m, n)

        # Straight-through: forward duro, backward por el suave
        gate = gate_hard - gate_soft.detach() + gate_soft

        w_eff = self.weight * gate
        return F.linear(x, w_eff, self.bias)

    @torch.no_grad()
    def gate_soft(self) -> torch.Tensor:
        return torch.sigmoid(self.logits / self.temperature)

    def sparsity_loss(self) -> torch.Tensor:
        """
        Penalización simple para empujar a gates a 0 (menos conexiones).
        Usar como: loss_total = loss + lam * layer.sparsity_loss()
        """
        gate_soft = torch.sigmoid(self.logits / self.temperature)
        return gate_soft.mean()  # promedio de conexiones “activas” (suaves)

    @torch.no_grad()
    def hard_mask(self) -> torch.Tensor:
        gate_soft = torch.sigmoid(self.logits / self.temperature)
        return (gate_soft > self.hard_threshold)

    @torch.no_grad()
    def export_edge_index_and_weights(self):
        """
        Devuelve (edge_index, edge_weight, bias) para construir una GraphLinear sparse real.
        """
        mask = self.hard_mask()  # (m,n) bool
        out_idx, in_idx = mask.nonzero(as_tuple=True)
        edge_index = torch.stack([out_idx, in_idx], dim=0)  # (2, E)
        edge_weight = (self.weight[out_idx, in_idx]).clone()
        bias = None if self.bias is None else self.bias.clone()
        return edge_index, edge_weight, bias

In [3]:
class VAE(nn.Module):
    def __init__(self, input_dim, output_dim, latent_dim=20):
        super(VAE, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.fc1 = LearnedSparseLinear(input_dim, 400)
        self.fc21 = LearnedSparseLinear(400, latent_dim)  # media
        self.fc22 = LearnedSparseLinear(400, latent_dim)  # logvar
        self.fc3 = LearnedSparseLinear(latent_dim, 400)
        self.fc4 = LearnedSparseLinear(400, output_dim)
    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc21(h1), self.fc22(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        # Quitamos sigmoid: los features de ResNet no están limitados a [0, 1]
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, self.input_dim))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [4]:
class DeMemte(nn.Module):
    def __init__(self, backbone, vae, embedding_dim, num_classes):
        super().__init__()
        self.backbone = backbone 
        self.vae = vae           
        
        self.feat_norm = nn.LayerNorm(embedding_dim)
        
        # Parámetro aprendible para escalar la sensibilidad de la memoria
        self.signal_gamma = nn.Parameter(torch.ones(1) * 0.1)
        
        # Peso del "recuerdo"
        self.memory_scale = nn.Parameter(torch.ones(1))
        
        # Clasificador final
        self.classifier = LearnedSparseLinear(embedding_dim, num_classes)

    def forward(self, x):
        # 1. Extraer características base y normalizar
        with torch.no_grad():
            raw_features = self.backbone(x)
        features = self.feat_norm(raw_features)
        
        # 2. Consultar memoria
        x_rec, mu, logvar = self.vae(features)
        
        # 3. Calcular "Confianza de la Memoria" (NUEVA FÓRMULA EXPONENCIAL)
        recon_diff = (features - x_rec)**2
        recon_loss_per_sample = torch.mean(recon_diff, dim=1, keepdim=True)
        
        # Usamos Softplus para asegurar que gamma sea positivo (>0)
        gamma = F.softplus(self.signal_gamma) 
        
        # Fórmula exponencial: 
        # Si Error -> 0, Signal -> 1.0 (Confianza total)
        # Si Error -> alto, Signal -> 0.0
        signal = torch.exp(-recon_loss_per_sample * gamma)
        
        # 4. Inyección de Memoria
        enhanced_features = features + (self.memory_scale * signal * x_rec)
        
        # 5. Clasificación
        logits = self.classifier(enhanced_features)
        
        return logits, x_rec, mu, logvar, features

    def train(self, mode: bool = True):
        super().train(mode)
        self.backbone.eval()
        return self

In [5]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [6]:
def train_dememte_v2(
    model, 
    trainloader, 
    num_epochs=15, 
    lr=1e-3, 
    lambda_sparsity=5e-3,  
    lambda_vae=1.0,        
    temp_start=2.0,        
    temp_end=0.5,          
    warmup_epochs=5,
    sleep_epochs=3 # Fase de pre-entrenamiento del VAE
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    # --- FASE 1: EL SUEÑO (Consolidación de Memoria) ---
    # En esta fase, el modelo aprende a reconstruir sin clasificar.
    print(f"--- Fase 0: Iniciando el Sueño ({sleep_epochs} épocas) ---")
    
    # Solo optimizamos el VAE y la normalización de entrada
    optimizer_sleep = torch.optim.Adam(
        list(model.vae.parameters()) + list(model.feat_norm.parameters()), 
        lr=lr
    )
    
    model.train()
    model.backbone.eval() # El backbone siempre en eval

    for s_epoch in range(sleep_epochs):
        running_recon = 0.0
        for inputs, _ in trainloader:
            inputs = inputs.to(device)
            optimizer_sleep.zero_grad()
            
            with torch.no_grad():
                raw_feats = model.backbone(inputs)
            
            features = model.feat_norm(raw_feats)
            x_rec, mu, logvar = model.vae(features)
            
            # Loss de reconstrucción pura + KL suave
            recon_loss = F.mse_loss(x_rec, features)
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / inputs.size(0)
            loss_sleep = recon_loss + 0.01 * kl_loss
            
            loss_sleep.backward()
            optimizer_sleep.step()
            running_recon += recon_loss.item()
            
        print(f"   [Sueño {s_epoch+1}/{sleep_epochs}] Recon Loss: {running_recon/len(trainloader):.4f}")

    # --- FASE 2: ENTRENAMIENTO CONJUNTO (Despertar) ---
    print("\n--- Fase 1: Entrenamiento Conjunto (DeMemte) ---")
    
    # Optimizador para TODO el modelo (incluyendo parámetros de señal)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion_task = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        # Programación de Sparsity y Temperatura
        if epoch < warmup_epochs:
            current_lambda_sparsity = 0.0
            current_temp = temp_start 
        else:
            current_lambda_sparsity = lambda_sparsity
            progress = (epoch - warmup_epochs) / max(1, num_epochs - warmup_epochs - 1)
            current_temp = temp_start + (temp_end - temp_start) * progress

        for m in model.modules():
            if hasattr(m, 'temperature'):
                m.temperature = float(current_temp)

        # Métricas de la época
        metrics = {k: 0.0 for k in ['Tsk', 'VAE', 'Sps', 'Acc', 'Sig']}
        active_params, total_params = 0, 0

        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            logits, x_rec, mu, logvar, features = model(inputs)

            # --- LOSSES ---
            loss_task = criterion_task(logits, labels)
            
            # Importante: Detach en features para que el VAE no intente "cambiar" 
            # las características de la ResNet, sino aprender a copiarlas.
            recon_loss = F.mse_loss(x_rec, features.detach()) 
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / inputs.size(0)
            loss_memory = recon_loss + 0.05 * kl_loss

            loss_sparse = torch.tensor(0.0, device=device)
            for m in model.modules():
                if hasattr(m, 'sparsity_loss'):
                    loss_sparse += m.sparsity_loss()

            total_loss = loss_task + (lambda_vae * loss_memory) + (current_lambda_sparsity * loss_sparse)

            total_loss.backward()
            optimizer.step()

            # --- LOGGING ---
            with torch.no_grad():
                metrics['Tsk'] += loss_task.item()
                metrics['VAE'] += loss_memory.item()
                metrics['Sps'] += loss_sparse.item()
                metrics['Acc'] += (logits.argmax(1) == labels).float().mean().item()
                
                gamma_val = F.softplus(model.signal_gamma)
                recon_err = torch.mean((features - x_rec)**2, dim=1, keepdim=True)
                metrics['Sig'] += torch.exp(-recon_err * gamma_val).mean().item()

            if i % 100 == 99:
                # Calcular densidad solo para el log
                current_active = sum(m.hard_mask().sum().item() for m in model.modules() if hasattr(m, 'hard_mask'))
                total_p = sum(m.hard_mask().numel() for m in model.modules() if hasattr(m, 'hard_mask'))
                
                print(f"[{epoch+1}, {i+1:5d}] "
                      f"Tsk: {metrics['Tsk']/100:.3f} | VAE: {metrics['VAE']/100:.3f} | "
                      f"Acc: {metrics['Acc']/100:.2f} | Sig: {metrics['Sig']/100:.4f} | "
                      f"Den: {100.*current_active/total_p:.1f}%")
                metrics = {k: 0.0 for k in metrics}

    print("Entrenamiento DeMemte finalizado.")

In [7]:
if __name__ == "__main__":
    import torchvision
    import torchvision.transforms as transforms
    import torchvision.models as models

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Trabajando en: {device}")

    # Transformaciones estándar para ResNet
    transform = transforms.Compose([
        transforms.Resize(224), 
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

    # 1. Configuración del Backbone (ResNet18)
    # Usamos pesos pre-entrenados para tener características ricas desde el inicio
    base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    backbone = torch.nn.Sequential(*list(base_model.children())[:-1], torch.nn.Flatten())
    
    # Congelar backbone: No queremos que la clasificación altere los rasgos base
    for p in backbone.parameters(): 
        p.requires_grad = False
    backbone.eval()
    
    # 2. Parámetros de arquitectura
    embedding_dim = 512
    num_classes = 10
    latent_dim = 128 # Dimensión latente de la memoria

    # 3. Instanciar Componentes
    # Sugerencia: El VAE actúa como un "autoencoder ruidoso" que limpia la señal
    memoria_vae = VAE(input_dim=embedding_dim, output_dim=embedding_dim, latent_dim=latent_dim)
    
    model = DeMemte(
        backbone=backbone,
        vae=memoria_vae,
        embedding_dim=embedding_dim,
        num_classes=num_classes
    )

    # 4. Ejecutar entrenamiento con fases
    train_dememte_v2(
        model=model,
        trainloader=trainloader,
        num_epochs=15,
        sleep_epochs=3,      # Fase de consolidación
        warmup_epochs=4,     # Espera antes de aplicar sparsity fuerte
        lr=1e-3,
        lambda_sparsity=1e-3, # Empezamos con sparsity suave
        lambda_vae=1.0
    )

Trabajando en: cuda


/home/giorgio6846/Code/nakato/Deseck/env/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


--- Fase 0: Iniciando el Sueño (3 épocas) ---
   [Sueño 1/3] Recon Loss: 0.2707
   [Sueño 2/3] Recon Loss: 0.0361
   [Sueño 3/3] Recon Loss: 0.0051

--- Fase 1: Entrenamiento Conjunto (DeMemte) ---
[1,   100] Tsk: 2.135 | VAE: 0.038 | Acc: 0.21 | Sig: 0.9919 | Den: 45.6%
[1,   200] Tsk: 1.197 | VAE: 0.226 | Acc: 0.57 | Sig: 0.9704 | Den: 44.4%
[1,   300] Tsk: 0.871 | VAE: 0.255 | Acc: 0.70 | Sig: 0.9746 | Den: 43.3%
[1,   400] Tsk: 0.754 | VAE: 0.239 | Acc: 0.75 | Sig: 0.9770 | Den: 42.5%
[1,   500] Tsk: 0.699 | VAE: 0.213 | Acc: 0.77 | Sig: 0.9777 | Den: 41.8%
[1,   600] Tsk: 0.645 | VAE: 0.183 | Acc: 0.79 | Sig: 0.9760 | Den: 41.0%
[1,   700] Tsk: 0.585 | VAE: 0.161 | Acc: 0.81 | Sig: 0.9727 | Den: 40.5%
[2,   100] Tsk: 0.523 | VAE: 0.133 | Acc: 0.83 | Sig: 0.9650 | Den: 39.6%
[2,   200] Tsk: 0.495 | VAE: 0.122 | Acc: 0.84 | Sig: 0.9608 | Den: 39.1%
[2,   300] Tsk: 0.486 | VAE: 0.117 | Acc: 0.84 | Sig: 0.9562 | Den: 38.6%
[2,   400] Tsk: 0.492 | VAE: 0.108 | Acc: 0.84 | Sig: 0.9520 |

In [8]:
# export complete model, no only state_dict
torch.save(model, "dememte_cifar10_complete.pth")

In [9]:
"""
class EmbeddingVAE(nn.Module):
    def __init__(self, H: int, latent_dim: int):
        super().__init__()
        self.fc1  = LearnedSparseLinear(H, 2*H)      # encoder trunk
        self.mu   = LearnedSparseLinear(2*H, latent_dim)
        self.logv = LearnedSparseLinear(2*H, latent_dim)
        self.fc3  = LearnedSparseLinear(latent_dim, 2*H)
        self.fc4  = LearnedSparseLinear(2*H, H)      # reconstruct h

    def encode(self, h):
        t = F.relu(self.fc1(h))
        return self.mu(t), self.logv(t)

    def reparameterize(self, mu, logv):
        std = torch.exp(0.5 * logv)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        t = F.relu(self.fc3(z))
        return self.fc4(t)  # sin sigmoid: embeddings pueden ser reales

    def forward(self, h):
        mu, logv = self.encode(h)
        z = self.reparameterize(mu, logv)
        h_hat = self.decode(z)
        return h_hat, mu, logv

"""

'\nclass EmbeddingVAE(nn.Module):\n    def __init__(self, H: int, latent_dim: int):\n        super().__init__()\n        self.fc1  = LearnedSparseLinear(H, 2*H)      # encoder trunk\n        self.mu   = LearnedSparseLinear(2*H, latent_dim)\n        self.logv = LearnedSparseLinear(2*H, latent_dim)\n        self.fc3  = LearnedSparseLinear(latent_dim, 2*H)\n        self.fc4  = LearnedSparseLinear(2*H, H)      # reconstruct h\n\n    def encode(self, h):\n        t = F.relu(self.fc1(h))\n        return self.mu(t), self.logv(t)\n\n    def reparameterize(self, mu, logv):\n        std = torch.exp(0.5 * logv)\n        eps = torch.randn_like(std)\n        return mu + eps * std\n\n    def decode(self, z):\n        t = F.relu(self.fc3(z))\n        return self.fc4(t)  # sin sigmoid: embeddings pueden ser reales\n\n    def forward(self, h):\n        mu, logv = self.encode(h)\n        z = self.reparameterize(mu, logv)\n        h_hat = self.decode(z)\n        return h_hat, mu, logv\n\n'